[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_7_deep_learning_6d_pose/18_intro_dl_for_cv.ipynb)

# Notebook 18 — Introduction to Deep Learning for Computer Vision

**Part 7: Deep Learning for 6D Pose** | Estimated time: 55 min

---

Parts 1–6 were **classical computer vision** — geometry, linear algebra, hand-crafted algorithms.  
Part 7 introduces **deep learning** — where the computer learns the geometry from data.

```
Classical CV path:          Deep learning path:

  Calibrate camera            Train or load model
       ↓                            ↓
  Find corners/features       Pass image through network
       ↓                            ↓
  Solve PnP                   Get pose directly
       ↓                            ↓
  Get R, t                    Get R, t

Deterministic, interpretable  Learned, flexible, no markers
```

## Recommended Watch

> Watch this **before** opening the notebook — it gives you context on why deep learning methods beat classical approaches for generalization to unknown objects.

| Title | Link | Duration |
|---|---|---|
| **3D Object Detection and Pose Estimation with Deep Learning in OpenCV Python** | [▶ Watch](https://youtu.be/R7zWFy7JmXc?si=dc5IrrhyiZrzGSQ4) | ~15 min |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install numpy matplotlib opencv-python -q
    print('Running in Google Colab — packages installed')
else:
    print('Running locally — make sure your venv is activated')

## Learning Objectives

By the end of this notebook you will be able to:

- Describe what a neural network is at an intuitive level
- Distinguish **training** from **inference** — and why this distinction matters for robotics
- Understand what **pretrained models** are and why they're so powerful
- Explain the GPU stack: CUDA → cuDNN → PyTorch/TensorFlow
- Know when to use classical CV vs. deep learning for 6D pose
- Name and compare the three DL approaches we'll cover in Part 7

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('Imports ready.')

## 1. What Is a Neural Network? (Intuition First)

A neural network is a **function approximator** — it learns to map inputs to outputs by adjusting millions of internal parameters.

**Analogy:** Think of a camera calibration — we had to carefully measure chessboard corners and solve equations to find K.  
A neural network instead says: *"Show me 10,000 examples of images + correct poses, and I'll figure out the mapping myself."*

```
Classical:
  [Image] → hand-craft features → geometry equations → [Pose]

Deep learning:
  [Image] → learned features (layers) → learned mapping → [Pose]
```

### The Building Block: A Neuron

A single neuron does:

$$y = \sigma(\mathbf{w}^T \mathbf{x} + b)$$

Where:
- $\mathbf{x}$ = input vector (e.g., pixel values)
- $\mathbf{w}$ = **weights** (learned parameters)
- $b$ = **bias** (also learned)
- $\sigma$ = **activation function** (adds non-linearity — the magic ingredient)
- $y$ = output

Stack millions of these → a neural network.

In [ ]:
# Simulate what a single neuron does
def relu(x):
    """ReLU activation: max(0, x)"""
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Example: neuron with 3 inputs
x = np.array([0.5, -0.3, 0.8])  # input
w = np.array([1.2, -0.5, 0.9])  # weights
b = 0.1                          # bias

z = w @ x + b                   # linear combination
y_relu    = relu(z)              # after ReLU
y_sigmoid = sigmoid(z)           # after sigmoid

print(f'Input x:        {x}')
print(f'Weights w:      {w}')
print(f'Bias b:         {b}')
print(f'Linear output z = w·x + b = {z:.4f}')
print(f'After ReLU:     {y_relu:.4f}  ← most common in modern nets')
print(f'After sigmoid:  {y_sigmoid:.4f} ← squashes to [0,1]')

# Visualize activation functions
x_plot = np.linspace(-4, 4, 300)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.plot(x_plot, relu(x_plot), 'b-', lw=2)
ax.set_title('ReLU: max(0, x)\n(most common in modern nets)', fontsize=10)
ax.axhline(0, c='gray', lw=0.5); ax.axvline(0, c='gray', lw=0.5)
ax.set_ylim(-0.5, 4); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x_plot, sigmoid(x_plot), 'g-', lw=2)
ax.set_title('Sigmoid: 1/(1+e^-x)\n(outputs [0,1], used for binary)', fontsize=10)
ax.axhline(0.5, c='gray', lw=0.5, linestyle='--'); ax.grid(True, alpha=0.3)

ax = axes[2]
tanh = np.tanh(x_plot)
ax.plot(x_plot, tanh, 'r-', lw=2)
ax.set_title('Tanh\n(outputs [-1,1])', fontsize=10)
ax.axhline(0, c='gray', lw=0.5); ax.grid(True, alpha=0.3)

plt.suptitle('Activation functions — the source of non-linearity', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Convolutional Neural Networks (CNNs) — The Backbone of Computer Vision

A regular neural network treats an image as a flat vector — terrible for images.  
A **CNN** uses **convolutional filters** that learn to detect local patterns:

```
Layer 1:  Detects edges (horizontal, vertical, diagonal)
Layer 2:  Combines edges → corners, curves
Layer 3:  Combines corners → object parts (wheels, handles, faces)
Layer N:  Combines parts → full object, class, pose
```

This **hierarchical feature learning** is why CNNs dominate image tasks.

```
Input image (H×W×3)
    │
    ├── Conv layer 1 → feature maps (H×W×64)
    │       ↓ pooling
    ├── Conv layer 2 → feature maps (H/2×W/2×128)
    │       ↓ pooling
    ├── Conv layer 3 → feature maps (H/4×W/4×256)
    │       ↓ pooling
    └── [Classification head] OR [Pose estimation head]
```

In [ ]:
import cv2

# Demonstrate a convolution filter manually
# This is what ONE learned filter in a CNN does

def apply_filter(image, kernel):
    """Apply a 2D convolution kernel to an image."""
    return cv2.filter2D(image, -1, kernel)

# Create a synthetic image
img = np.zeros((120, 200), dtype=np.float32)
img[30:90, 50:150] = 1.0  # white rectangle

# Different learned filters at different layers
filters = {
    'Edge (horizontal)':  np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=np.float32),
    'Edge (vertical)':    np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32),
    'Blur (smoothing)':   np.ones((5,5), dtype=np.float32) / 25,
    'Sharpen':            np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32),
}

fig, axes = plt.subplots(1, len(filters)+1, figsize=(18, 4))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Input image', fontsize=10)
axes[0].axis('off')

for ax, (name, kernel) in zip(axes[1:], filters.items()):
    result = apply_filter(img, kernel)
    ax.imshow(result, cmap='gray')
    ax.set_title(f'{name}\n{kernel.shape} kernel', fontsize=9)
    ax.axis('off')

plt.suptitle('Convolution filters — in a CNN these kernels are LEARNED, not hand-crafted',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('In a CNN:')
print('  Layer 1 → learns edge detectors like these')
print('  Layer 5 → learns texture patterns')
print('  Layer 15 → learns object parts (wheel, handle, corner)')
print('  Layer 50 → learns complete object representations')

## 3. Training vs. Inference — The Critical Distinction for Robotics

This is the most important concept for using DL in production robotics.

```
TRAINING                           INFERENCE
──────────────────────────────     ─────────────────────────────────
Done ONCE (offline)               Done at runtime (on robot)
Needs thousands of labeled images  Needs ONE image
Takes hours or days on GPU         Takes milliseconds
High memory usage                  Low memory (optimized model)
Researcher/ML engineer does this   Robot software engineer does this
Produces: model weights file       Produces: prediction (pose)
```

**For robotics engineers:** You almost never train from scratch. You use **pretrained models** and run inference.

In [ ]:
# Simplified illustration of what a training loop looks like
# We train a tiny network to approximate sin(x)

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Tiny 2-layer network --- 
class TinyNet:
    """A minimal neural network with 1 hidden layer."""
    def __init__(self, n_hidden=16):
        self.W1 = np.random.randn(1, n_hidden) * 0.1
        self.b1 = np.zeros(n_hidden)
        self.W2 = np.random.randn(n_hidden, 1) * 0.1
        self.b2 = np.zeros(1)
    
    def forward(self, x):
        self.x = x
        self.z1 = x @ self.W1 + self.b1
        self.a1 = np.tanh(self.z1)               # hidden layer
        self.z2 = self.a1 @ self.W2 + self.b2
        return self.z2                            # output (no activation)
    
    def backward(self, y_true, lr=0.01):
        y_pred = self.z2
        n = len(y_true)
        # Backprop
        dL_dz2 = 2 * (y_pred - y_true) / n
        dW2 = self.a1.T @ dL_dz2
        db2 = dL_dz2.sum(axis=0)
        da1 = dL_dz2 @ self.W2.T
        dz1 = da1 * (1 - self.a1**2)   # tanh derivative
        dW1 = self.x.T @ dz1
        db1 = dz1.sum(axis=0)
        # Gradient descent step
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1

# --- Training ---
X = np.linspace(-np.pi, np.pi, 200).reshape(-1, 1)
y = np.sin(X)

net = TinyNet(n_hidden=32)
losses = []
snapshots = {0: None, 10: None, 100: None, 500: None}

for epoch in range(501):
    y_pred = net.forward(X)
    loss = np.mean((y_pred - y)**2)
    losses.append(loss)
    if epoch in snapshots:
        snapshots[epoch] = y_pred.copy()
    net.backward(y, lr=0.05)

# --- Plot training progress ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(losses, 'b-', lw=1.5)
axes[0].set_xlabel('Training epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Training loss decreases as network learns', fontsize=11)
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

x_plot = X.flatten()
colors = ['red', 'orange', 'blue', 'green']
for (epoch, pred), color in zip(snapshots.items(), colors):
    axes[1].plot(x_plot, pred.flatten(), '--', color=color,
                 alpha=0.8, label=f'Epoch {epoch}')
axes[1].plot(x_plot, np.sin(x_plot), 'k-', lw=2, label='True sin(x)')
axes[1].set_title('Network approximates sin(x) over training', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training = adjusting millions of weights to minimize loss',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Final loss after 500 epochs: {losses[-1]:.6f}')
print(f'Network parameters: {32 + 32 + 32 + 1} (tiny example)')
print(f'Real pose networks: millions to billions of parameters')

## 4. Pretrained Models — Standing on Giants' Shoulders

Training a vision model from scratch requires:
- Millions of labeled images
- Weeks of GPU time
- Significant ML expertise

**Pretrained models** are trained once by researchers and shared publicly.  
You download the weights and use them immediately.

### Transfer Learning

Even better — you can **fine-tune** a pretrained model on your own data:

```
ImageNet pretrained backbone
  (already knows: edges, textures, shapes, objects)
         │
         ▼ Add your head:
  [Pose estimation head for YOUR robot part]
         │
         ▼ Fine-tune on 100-1000 labeled images
         │
         ▼ Working pose estimator in hours, not weeks
```

| Strategy | Data Needed | Training Time | Use When |
|---|---|---|---|
| Use pretrained as-is | None | None | Object is in supported classes |
| Fine-tune last layers | 100–1000 examples | Hours | Custom object, similar domain |
| Fine-tune full model | 1K–10K examples | Days | Custom object, very different domain |
| Train from scratch | 100K+ examples | Weeks | Completely new problem |

For the notebooks in Part 7, we **always use pretrained models** — no training required.

## 5. The GPU Stack — Why GPUs Matter

Neural network inference is just a massive amount of **matrix multiplications**.  
GPUs have thousands of cores designed exactly for parallel math.

```
CPU: 8–32 big cores → good for sequential logic
GPU: 3000–10000 small cores → perfect for matrix math

ResNet-50 inference:
  CPU: ~150ms per image → ~7 FPS
  GPU: ~5ms per image → ~200 FPS
```

### The Software Stack

```
Your code (PyTorch / TensorFlow)
         │
      cuDNN  ← NVIDIA's deep learning primitives (conv, RNN, etc.)
         │
      CUDA   ← NVIDIA's GPU programming platform
         │
  GPU Hardware (NVIDIA RTX, A100, etc.)
```

**Critical constraint:** PyTorch, CUDA, and cuDNN versions must be **compatible**.  
Using a matrix from https://pytorch.org/get-started/locally/ is essential.

In [ ]:
# Check GPU availability
import subprocess

def check_gpu_status():
    print('=== GPU Status Check ===')
    print()
    
    # nvidia-smi
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
             '--format=csv,noheader'],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0:
            lines = result.stdout.strip().split('\n')
            for i, line in enumerate(lines):
                name, mem, driver = [x.strip() for x in line.split(',')]
                print(f'GPU {i}: {name}')
                print(f'  Memory: {mem}')
                print(f'  Driver: {driver}')
        else:
            print('nvidia-smi not found → no NVIDIA GPU or driver not installed')
    except (FileNotFoundError, subprocess.TimeoutExpired):
        print('nvidia-smi not available → likely no NVIDIA GPU')
    
    print()
    
    # PyTorch GPU check
    try:
        import torch
        print(f'PyTorch version: {torch.__version__}')
        print(f'CUDA available: {torch.cuda.is_available()}')
        if torch.cuda.is_available():
            print(f'CUDA version: {torch.version.cuda}')
            print(f'GPU count: {torch.cuda.device_count()}')
            for i in range(torch.cuda.device_count()):
                props = torch.cuda.get_device_properties(i)
                print(f'  GPU {i}: {props.name} | {props.total_memory/1e9:.1f} GB')
        else:
            print('→ Running on CPU only')
    except ImportError:
        print('PyTorch not installed — install with:')
        print('  pip install torch torchvision  (CPU)')
        print('  OR visit https://pytorch.org/get-started/locally/ for GPU version')

check_gpu_status()

## 6. The Three Deep Learning Approaches We'll Cover

Part 7 covers three distinct DL approaches to 6D pose, ordered from easiest to most powerful:

| Notebook | Method | Need GPU? | Needs Training? | Works on Custom Objects? |
|---|---|---|---|---|
| NB 19 | MediaPipe Objectron | No (CPU) | No (pretrained) | No (fixed classes only) |
| NB 20 | EfficientPose | Yes (recommended) | No (pretrained) | Limited (LineMOD classes) |
| NB 21 | FoundationPose / FreeZe | Yes | No (zero-shot) | Yes (any object + CAD) |
| NB 22 | MegaPose + ViSP | Yes | No (zero-shot) | Yes (CAD model) |

**The trade-off spectrum:**

```
Easy to use ────────────────────────────────────────── Flexible
│                                                               │
MediaPipe          EfficientPose         FoundationPose/FreeZe  │
CPU, 3 objects     GPU, more objects     GPU, ANY object + CAD   │
│                                                               │
Constrained ───────────────────────────────────────── Powerful
```

### When Deep Learning Beats Classical

| Situation | Use Classical | Use Deep Learning |
|---|---|---|
| Can print a marker | ✅ ArUco — fast, robust | Overkill |
| Need 0.1mm accuracy | ✅ solvePnP + calibrated camera | Typically not accurate enough |
| No marker, textureless object | ❌ Hard to detect | ✅ DL handles textureless |
| Need to work from day 1 | ✅ Classical, no data needed | ❌ Needs training data |
| Many different objects | ❌ Need separate solution per object | ✅ One model, many objects |
| Clutter / partial occlusion | ❌ Classical struggles | ✅ DL is more robust |

In [ ]:
# Visualize the method comparison
methods = [
    {'name': 'solvePnP\n(Part 4)',    'ease': 9, 'accuracy': 9, 'flexibility': 3, 'color': '#2196F3'},
    {'name': 'ArUco\n(Part 5)',       'ease': 10,'accuracy': 8, 'flexibility': 4, 'color': '#4CAF50'},
    {'name': 'Stereo\n(Part 6)',      'ease': 6, 'accuracy': 7, 'flexibility': 6, 'color': '#9C27B0'},
    {'name': 'MediaPipe\n(NB 19)',    'ease': 10,'accuracy': 5, 'flexibility': 2, 'color': '#FF9800'},
    {'name': 'EfficientPose\n(NB 20)','ease': 7, 'accuracy': 7, 'flexibility': 5, 'color': '#F44336'},
    {'name': 'FoundationPose\n(NB 21)','ease': 5,'accuracy': 8, 'flexibility': 9, 'color': '#795548'},
    {'name': 'FreeZe/MegaPose\n(NB 21-22)','ease': 4,'accuracy': 8,'flexibility': 10,'color':'#607D8B'},
]

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
labels = [m['name'] for m in methods]
colors = [m['color'] for m in methods]
x_pos = np.arange(len(methods))

for ax, metric in zip(axes, ['ease', 'accuracy', 'flexibility']):
    vals = [m[metric] for m in methods]
    bars = ax.bar(x_pos, vals, color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=7.5)
    ax.set_ylim(0, 11)
    ax.set_yticks(range(0, 11, 2))
    ax.set_ylabel('Score (1-10)', fontsize=10)
    title = {'ease': 'Ease of Use', 'accuracy': 'Pose Accuracy', 'flexibility': 'Object Flexibility'}[metric]
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.2, str(val),
                ha='center', va='bottom', fontsize=8)

plt.suptitle('6D Pose Method Comparison — no single winner, choose by use case',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. The General Inference Workflow

Every DL pose estimation follows this structure:

```python
# 1. Load model (weights file)
model = load_model('pose_model.pth')  # or .h5, .pb, .onnx
model.eval()  # switch off training-specific layers (dropout, batchnorm)

# 2. Preprocess input
frame = capture_frame()
input_tensor = preprocess(frame)  # resize, normalize, add batch dim

# 3. Run inference
with torch.no_grad():  # skip gradient computation (faster)
    outputs = model(input_tensor)

# 4. Post-process output
rvec, tvec = decode_pose(outputs)  # model-specific decoding

# 5. Use the pose
send_to_robot_controller(rvec, tvec)
```

The main variation between notebooks 19–22 is steps 1, 3, and 4.

## Recap

| Concept | Key Point |
|---|---|
| Neural network | Learns to map inputs → outputs by adjusting weights |
| CNN | Learns hierarchical visual features: edges → parts → objects |
| Training | Done once, offline, needs labeled data + GPU, produces weights |
| Inference | Runtime, fast, uses loaded weights to predict pose |
| Pretrained model | Download weights, run inference immediately — no training needed |
| Transfer learning | Fine-tune pretrained backbone on small custom dataset |
| GPU vs CPU | GPU: 10–100× faster for matrix ops — essential for real-time inference |
| CUDA stack | Hardware → CUDA → cuDNN → PyTorch/TF → your code |

**Next:** MediaPipe Objectron — 30+ FPS 3D object detection on CPU.

---
## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Manual forward pass
# ============================================================
# Implement a 3-layer neural network forward pass manually.
# Layer sizes: 4 → 8 → 4 → 2 (input dim=4, output dim=2)
# Use ReLU for hidden layers, no activation on output.
# Generate random weights and pass x = [0.1, 0.5, -0.3, 0.8].
# Print the final output.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# np.random.seed(0)
# W1 = np.random.randn(4, 8) * 0.1; b1 = np.zeros(8)
# W2 = np.random.randn(8, 4) * 0.1; b2 = np.zeros(4)
# W3 = np.random.randn(4, 2) * 0.1; b3 = np.zeros(2)
# x = np.array([0.1, 0.5, -0.3, 0.8])
# a1 = np.maximum(0, x @ W1 + b1)   # ReLU
# a2 = np.maximum(0, a1 @ W2 + b2)  # ReLU
# y  = a2 @ W3 + b3                  # linear output
# print(f'Output: {y.round(4)}')
# print(f'Layer sizes: {x.shape} → {a1.shape} → {a2.shape} → {y.shape}')

In [ ]:
# ============================================================
# EXERCISE 2: Method selection
# ============================================================
# For each scenario, recommend the best method (ArUco, solvePnP,
# MediaPipe, EfficientPose, or FreeZe/FoundationPose) and explain why.
#
# Scenarios:
# A) Docking a robot to a charging station you control (can add markers)
# B) Detecting random packages on a conveyor (unknown shapes, textureless)
# C) Tracking a coffee cup for an AR demo on CPU (no GPU available)
# D) Industrial bin picking with 50 different part types (have CAD models)
# Write your answers as print statements.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# print('A) ArUco — you control the environment, can print markers,')
# print('   deterministic, accurate, fast. No DL needed.')
# print()
# print('B) FreeZe or FoundationPose — textureless objects, unknown shapes,')
# print('   zero-shot with CAD model. Classical methods will struggle.')
# print()
# print('C) MediaPipe Objectron — CPU-only, real-time, cup is a supported class.')
# print()
# print('D) FreeZe/MegaPose — zero-shot with CAD models, handles many object types,')
# print('   no per-object training needed. One model, 50 parts.')